# Exploration of Online Retail Data
The block below reads the raw data file and drops the records pertaining to:
1. Records having missing values
2. Invoice line items pertaining to bank charges
3. Cancelled transactions

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
os.chdir("/home/admin123/Starspace/")
fp = "/home/admin123/Starspace/"+ "data/Online_Retail.csv"
df = pd.read_csv(fp)
df = df[-df['InvoiceNo'].str.startswith("C")]
df = df[-df['StockCode'].str.startswith("BANK")]
#req_cols = ["CustomerID", "InvoiceNo", "StockCode"]
df = df.dropna(how ='any')
df["CustomerID"] = df["CustomerID"].astype(int)
#df = df.sort_values(by=['CustomerID'])

### Verify that no records with missing values exist

In [ ]:
df[pd.isnull(df).any(axis=1)].shape
#df.query("CustomerID == 18198")

## Description of the Data
Each line of the data file represents a line item in an invoice. Some attributes are added for analysis. Here is a snapshot of the data in the raw format (as obtained). 

In [ ]:
df.head()

### Attribute Addition:
The total cost associated with each line item is $ quantity \times unit\ price$. This is missing in the raw data. We create an attribute for this. It helps with the analysis.

In [ ]:
df["ItemTotal"] = df["UnitPrice"] * df["Quantity"]

In [ ]:
df.head()

## Calculate Customer Characteristics
We want to charactrize the shopping habbits of the customers. We want to know:
1. How much do they spend when they make a purchase?
2. How often do they visit the online store?
3. What kind of things do they purchase and how many at a time of each?
4. How much has the customer spent at the store over the period captured by the data?
5. What is a typical spend at the store?
6. Who are the valuable customers and what do they buy?


To answer questions like this, we will restructure the raw data to answer these questions. We will group the data by customer and the invoice, i.e., for each customer we will capture their invoices. For each invoice, we will calculate the total spend and the total number of line items

### Agrregate Invoice Line Items per Customer and Invoice
We will use the group by construct with pandas to group invoice line items by customer and invoice. The result would order records by customer. For each customer the data is ordered by the invoices associated with him/her. For each invoice, we will determine the number of items purchased, the total spend and the number of line items in the invoice. The number of line items in the invoice gives us a sense of the variety of things the customer purchased in that visit. 

Let's start with aggregating the invoices for each customer. This is shown below. In the aggregation below:


- StockCode count: represents the number of distinct items in the invoices. For example, if the invoice lists item A and item B only. Then the StockCode count is 2


- ItemTotal sum: This is the amount the customer spent on that invoice. It is sum of the itemTotals.


- Quantity sum: This is the total number of items associated with that invoice. For example if the invoice has 3 of item A and 2 of item B, the Quantity sum is 5.



In [ ]:
df_inv_summary = df.groupby(["CustomerID","InvoiceNo"]).\
agg({'ItemTotal': ['sum'], 'Quantity':['sum'], 'StockCode':['count']}).reset_index()

In [ ]:
df_inv_summary.columns = [''.join(col).strip() for col in df_inv_summary.columns]

## Sample of Invoice Aggregation
See a sample of the invoice aggregation below. Interpretation: On invoice 537626, Customer 12347 spent 711.79. The purchase involved 31 different types of items. 319 items (total quantity) were purchased as part of that invoice.

In [ ]:
df_inv_summary.head()

## Summarization of the Invoices
A summary view of customer invoices is shown below. The mean associated with the invoice summary gives us the average values for the attributes. As is evident from reviewing the summary below, the data has some outliers. The mean values for the StockCode count, invoice total and the item quantity count is very different from the maximum values for these attributes. A typical purchase is fairly small, but we have some outliers. Histograms for the Stockcode count, invoice total and the item quantity confirm this skewed behavior. Most of the invoices lie in the left region of these histogram plots.

In [ ]:
df_inv_summary.describe()

In [ ]:
df_inv_summary["StockCodecount"].hist()

In [ ]:
df_inv_summary["ItemTotalsum"].hist()

## Aggregate Customers
Let us analyze customers next. To do so we need to aggregate by Customer ID and count invoices and sum the invoice totals. This will tell us the total amount spent by the customer and the number of store visits 

In [ ]:

df_freq_cust = df_inv_summary.groupby('CustomerID').\
agg({'ItemTotalsum': ['sum'], 'InvoiceNo':['count']}).reset_index()

## Sample of Customer Aggregation

In [ ]:
df_freq_cust.columns = [''.join(col).strip() for col in df_freq_cust.columns]
df_freq_cust.columns = ["CustomerID", "Store_Visits", "Amount_Spent"]
df_freq_cust.head()

## Summarize Customer View

In [ ]:
df_freq_cust.describe()

## Observations About Customers
- The store enjoys customer loyalty. On average, a customer generates about 5 invoices at the store.
- There are some customer outliers - some very atypical customers. This can be identified by inspecting the maximum values for the store visits and the amount spent

We will confirm these with histogram views for these attributes

In [ ]:
df_freq_cust["Store_Visits"].hist()

In [ ]:
df_freq_cust["Amount_Spent"].hist()

### A View of Atypical Customers
Let's take a look at customers who are atypical. We will define an atypical customer to be one that spends over 3 standard deviations from the mean customer spend at the store. We will first define this threshold


In [ ]:
atypical_threshold = df_freq_cust["Amount_Spent"].mean() + 3 * df_freq_cust["Amount_Spent"].std()
df_ob = df_freq_cust[df_freq_cust.Amount_Spent > atypical_threshold]

## Sample of Atypical Customers

In [ ]:
df_ob.head()

## Histogram View of Atypical Customers

In [ ]:
df_ob["Amount_Spent"].hist()

In [ ]:
df_ob["Store_Visits"].hist()

## Observations
- Both histograms are skewed and captures the atypical customers. For example there is a customer who spends over 250 K at the store and there is a customer who has generated over 200 invoices at the store.

In [ ]:
df.columns

In [ ]:
df.groupby("CustomerID")